In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from collections import Counter
import re

base_url = "https://te.eg"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

NOISE_KEYWORDS = [
    "Copyright", "حقوق النشر", "NTRA", "الرجوع الي الأعلي",
    "Return To Top", "تواصل معنا", "حمل التطبيق",
    "الفروع", "أسئلة متكرره", "اتصل بنا", "155"
]

def is_noise(text):
    return any(keyword in text for keyword in NOISE_KEYWORDS)


def discover_service_links(category_url):
    """Get all sub-service links from a category page"""
    response = requests.get(category_url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")

    links = set()
    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"].strip()
        if "/w/" in href:
            links.add(urljoin(base_url, href))
    return links


def extract_raw_lines(page_url):
    """Get all unique text lines from a single page (dedup within the same page)"""
    response = requests.get(page_url, headers=headers)
    if response.status_code != 200:
        return []
    soup = BeautifulSoup(response.content, "html.parser")

    lines = []
    seen = set()  # dedup lines within this page
    for tag in soup.find_all(['h1', 'h2', 'h3', 'p', 'li']):
        text = re.sub(r'\s+', ' ', tag.get_text(strip=True, separator=' '))
        if len(text) > 10 and text not in seen:
            lines.append(text)
            seen.add(text)
    return lines


def build_boilerplate_set(pages_raw_lines):
    """Lines repeated across more than one page are nav/footer boilerplate"""
    counter = Counter()
    for lines in pages_raw_lines:
        counter.update(set(lines))
    return {line for line, count in counter.items() if count > 1}


def chunk_page(content, max_chars=500, min_chars=50):
    """Split page content into chunks, keeping sentences intact.
    Merges a too-short trailing chunk into the previous one instead of
    leaving an orphan chunk."""
    sentences = re.split(r'(?<=[.!؟?])\s+', content)

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chars:
            current_chunk += (" " if current_chunk else "") + sentence
        else:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = sentence

    if current_chunk:
        chunks.append(current_chunk)

    # Merge a short trailing chunk into the previous one
    if len(chunks) > 1 and len(chunks[-1]) < min_chars:
        chunks[-2] = chunks[-2] + " " + chunks[-1]
        chunks.pop()

    return chunks

In [ ]:
category_url = "https://te.eg/web/guest/personal/services/entertainment"
service_links = discover_service_links(category_url)

# Step 1: collect raw lines from every page first
all_pages_lines = {link: extract_raw_lines(link) for link in service_links}

# Step 2: find lines repeated across pages -> treat as boilerplate
boilerplate = build_boilerplate_set(all_pages_lines.values())

# Step 3: build knowledge_base keeping only clean, unique-per-page lines
knowledge_base = []
for url, lines in all_pages_lines.items():
    clean_lines = [l for l in lines if l not in boilerplate and not is_noise(l)]
    if clean_lines:
        knowledge_base.append({"url": url, "content": " ".join(clean_lines)})

print(f"Pages collected: {len(knowledge_base)}")

# Step 4: chunk every page
final_chunks = []
for item in knowledge_base:
    page_chunks = chunk_page(item["content"])
    for i, chunk_text in enumerate(page_chunks):
        final_chunks.append({
            "chunk_id": f"{item['url'].split('/')[-1]}_{i}",
            "url": item["url"],
            "text": chunk_text,
            "category": "entertainment"
        })

print(f"Total chunks created: {len(final_chunks)}")
for c in final_chunks:
    print("---")
    print(c["chunk_id"], "|", c["text"])

Pages collected: 5
Total chunks created: 12
---
mobile-number-portability_0 | كيف يمكن التحويل إلى WE ؟
---
mobile-number-portability_1 | عليك إحضار تحقيق الشخصية الخاص بك و التأكد من صحة البيانات في اقرب فرع WE بطاقه الرقم القومي سارية وبلا أي كسور أو تلفيات قد تستقبل أي فواتير بمبالغ مستحقة من الشركة السابقة خلال أول 60 يوم تأكيد طلب التحويل عند استلام المكالمة الهاتفية إنهاء جميع المستحقات المالية في شركة الموبايل التي ترغب في الإنتقال منها قبل التحويل عليك إحضار تحقيق الشخصية الخاص بك و التأكد من صحة البيانات في اقرب فرع WE بطاقه الرقم القومي سارية وبلا أي كسور أو تلفيات قد تستقبل أي فواتير بمبالغ مستحقة من الشركة السابقة خلال أول 60 يوم تأكيد طلب التحويل عند استلام المكالمة الهاتفية إنهاء جميع المستحقات المالية في شركة الموبايل التي ترغب في الإنتقال منها قبل التحويل ما هي مصاريف التحويل ؟
---
mobile-number-portability_2 | مصاريف التحويل مجانية و يتم إستلام الشريحة الجديدة من WE مجاناً مصاريف التحويل مجانية و يتم إستلام الشريحة الجديدة من WE مجاناً كم من الوقت يستغرق لتنفيذ الطلب؟ 

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import json

embedding_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

chunk_texts = [c["text"] for c in final_chunks]

print("Generating embeddings...")
chunk_embeddings = embedding_model.encode(chunk_texts, show_progress_bar=True)

print(f"Embeddings shape: {chunk_embeddings.shape}")  # (num_chunks, 384)

# Save the vectors
np.save('chunk_embeddings.npy', chunk_embeddings)

# Save the matching metadata (chunk_id, url, text, category) — order must match the embeddings array
with open('chunks_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(final_chunks, f, ensure_ascii=False, indent=2)

print("Saved embeddings + metadata")



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (12, 384)
Saved embeddings + metadata


In [ ]:
!pip install -q chromadb

import chromadb

client = chromadb.Client()

# Delete if it already exists from a previous run, then recreate clean
try:
    client.delete_collection(name="we_entertainment")
except Exception:
    pass

collection = client.create_collection(
    name="we_entertainment",
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    ids=[c["chunk_id"] for c in final_chunks],
    embeddings=chunk_embeddings.tolist(),
    documents=[c["text"] for c in final_chunks],
    metadatas=[{"url": c["url"], "category": c["category"]} for c in final_chunks]
)

print(f"Added {collection.count()} chunks to the vector database")


def search(query, top_k=3):
    query_embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    return results




In [ ]:
!pip install -q google-genai
from google import genai as genai_client
from google.colab import userdata

client_gemini = genai_client.Client(api_key=userdata.get('GOOGLE_API_KEY'))

In [ ]:
SYSTEM_PROMPT = """You are a customer support assistant for Telecom Egypt (WE).\nAnswer ONLY using the information provided in the context below.\nIf the answer is not in the context, say clearly that you don't have this information and suggest contacting WE customer service.\nAlways answer in the same language as the customer's question (Arabic or English).\nBe concise and include exact prices, USSD codes, or SMS numbers when relevant."""

In [ ]:
def generate_answer(query, top_k=3):
    results = search(query, top_k=top_k)
    context = "\n\n".join(results["documents"][0])
    prompt = f"{SYSTEM_PROMPT}\n\nContext:\n{context}\n\nCustomer question: {query}"

    response = client_gemini.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=prompt
    )
    return response.text

In [ ]:
import gradio as gr

WE_PURPLE = "#5C2D91"
WE_PURPLE_LIGHT = "#7B4FA8"

custom_css = f"""
.gradio-container {{
    font-family: 'Segoe UI', Tahoma, sans-serif !important;
    direction: rtl;
}}

#header-title {{
    color: {WE_PURPLE};
    text-align: center;
    font-weight: 700;
}}

.message.user {{
    background-color: {WE_PURPLE} !important;
    color: white !important;
    border-radius: 16px !important;
}}

.message.bot {{
    background-color: #F5F0FA !important;
    color: {WE_PURPLE} !important;
    border-radius: 16px !important;
    border: 1px solid {WE_PURPLE_LIGHT} !important;
}}

.message.bot * {{
    color: {WE_PURPLE} !important;
}}

button.primary {{
    background-color: {WE_PURPLE} !important;
    border: none !important;
}}

button.primary:hover {{
    background-color: {WE_PURPLE_LIGHT} !important;
}}
"""

def chat_interface(message, history):
    return generate_answer(message)

with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="purple")) as demo:
    gr.Markdown(
        "<h1 id='header-title'>خدمة عملاء WE 🟣</h1>"
        "<p style='text-align:center; color:#666;'>اسأل عن خدمات الترفيه — WATCH IT، كول تون، WE Sports، العب واكسب</p>"
    )

    gr.ChatInterface(
        fn=chat_interface,
        examples=[
            "عايز اشترك في واتش إت، التكلفة كام؟",
            "عايز اعرف سعر الكول تون",
            "فيه خدمة لمتابعة أخبار الكورة؟",
            "عايز العب واكسب فلوس، ازاي اشترك؟",
        ],
    )

demo.launch(inline=True, share=True)

/tmp/ipykernel_8326/843856372.py:48: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="purple")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0b2374fa488ee06688.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
